# Executor do experimento no ambiente do Google Colab

## Verificação do ambiente de execução

- Valida a GPU
- Documenta o ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sat Apr 25 00:45:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

## Clonagem do repositório do GitHub

- Clona o repositório via `git clone`
- Transfere o ambiente de execução para a pasta raiz do repositório

In [1]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn
!git checkout develop

/content
Cloning into 'classification-of-medical-images-using-cnn'...
remote: Enumerating objects: 546, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 546 (delta 142), reused 219 (delta 94), pack-reused 269 (from 1)
Receiving objects: 100% (546/546), 6.81 MiB | 12.19 MiB/s, done.
Resolving deltas: 100% (275/275), done.
/content/classification-of-medical-images-using-cnn
Branch 'develop' set up to track remote branch 'develop' from 'origin'.
Switched to a new branch 'develop'


## Montagem do Google Drive

- Monta o Google Drive permitindo que os arquivos presentes nele sejam lidos e escritos
- Define o diretório base de onde serão carregados os dados de treino, validação e teste, e onde serão salvos o modelo e os resultados do experimento

In [2]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Mounted at /content/drive


## Definição de parâmetros gerais

- `EXPERIMENT_NAME`: nome do experimento, neste caso, se tratando de um experimento de teste
- `IMAGE_SIZE`: tamanho para o qual as imagens serão resimensionadas para que o modelo possa analisar
- `BATCH_SIZE`: tamanho do lote de imagens que serão analisadas pelo modelo a cada interação
- `EPOCHS`: número de vezes que o modelo analisará todas as imagens do conjunto de treinamento
- `SEED`: indica como será feito o embaralhamento do conjunto de dados

In [3]:
EXPERIMENT_NAME = "test"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
SEED = 42
RUN_ID = 1

## Treinamento do modelo

- Impporta o pipeline de treinamento
- Executa o treinamento e salva o modelo e o histótico de treino na pasta do Google Drive

In [5]:
from src.train import train_pipeline

config = train_pipeline(
    base_dir=BASE_PATH,
    expereriment_name=EXPERIMENT_NAME,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    run_id=RUN_ID
)

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 682s 4s/step - AUC: 0.7123 - accuracy: 0.7400 - loss: 0.5294 - val_AUC: 0.8047 - val_accuracy: 0.5625 - val_loss: 0.7622
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 13s 78ms/step - AUC: 0.8533 - accuracy: 0.7828 - loss: 0.4361 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7183
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 13s 79ms/step - AUC: 0.8730 - accuracy: 0.8075 - loss: 0.3970 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7484
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 13s 80ms/step - AUC: 0.8870 - accuracy: 0.8225 - loss: 0.3738 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7668
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 13s 80ms/step - AUC: 0.8961 - accuracy: 0.8330 - loss: 0.3584 - val_AUC: 0.8125 - val_accuracy: 0.6250 - val_loss: 0.7820
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 13s 81ms/step - AUC:

In [7]:
import json

print(json.dumps(config, indent=4))

{
    "base_model": "ResNet50",
    "weights": "imagenet",
    "optimizer": {
        "name": "adam",
        "learning_rate": 0.001
    },
    "preprocessing": [
        "rescaling",
        "data augmentation"
    ],
    "image_size": [
        224,
        224
    ],
    "batch_size": 32,
    "class_weights": false,
    "epochs": 10,
    "seed": 42
}


In [6]:
from src.test import test_pipeline

metrics = test_pipeline(
    base_dir=BASE_PATH,
    experiment_name=EXPERIMENT_NAME,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    run_id=RUN_ID
)

Found 624 files belonging to 2 classes.
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 263ms/step

 Experiment Summary (test)
+----------------------+------------+-------------+----------+------------+---------------+-----------+
|   Decision Threshold |   Accuracy |   Precision |   Recall |   F1-Score |   Specificity |   AUC ROC |
+======================+============+=============+==========+============+===============+===========+
|                  0.5 |   0.791667 |    0.776596 | 0.935897 |   0.848837 |      0.551282 |  0.885152 |
+----------------------+------------+-------------+----------+------------+---------------+-----------+


In [8]:
print(json.dumps(metrics, indent=4))

{
    "decision-threshold": 0.5,
    "accuracy": 0.7916666666666666,
    "precision": 0.776595744680851,
    "recall": 0.9358974358974359,
    "f1-score": 0.8488372093023255,
    "specificity": 0.5512820512820513,
    "auc-roc": 0.885152312075389
}


In [4]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/experiments.db"

engine = get_engine(DB_PATH)

In [6]:
insert_run(
    engine=engine,
    experiment=EXPERIMENT_NAME,
    run_name=f"run_{RUN_ID}",
    config=config,
    metrics=metrics,
)

1